### **.pkl** model load and information checking

In [1]:
MODEL_FOLDER = "../ThirdParty/Strategies/models/"

In [2]:
import joblib
pkl_model_path = MODEL_FOLDER + "rf_model.pkl"
rf_model = joblib.load(pkl_model_path)

# basic info
print("n_features_in:", rf_model.n_features_in_)
print("n_estimators:", rf_model.n_estimators)
print("max_depth:", rf_model.max_depth)
print("min_samples_leaf:", rf_model.min_samples_leaf)

# classes and counts
print("classes:", rf_model.classes_)
print("n_classes_:", rf_model.n_classes_)

# feature importances (show top 10)
import numpy as np
fi = rf_model.feature_importances_
print("feature_importances (len={}):".format(len(fi)))
top_idx = np.argsort(fi)[::-1]
for i in top_idx[:10]:
    print(f"  feature {i}: importance={fi[i]:.4f}")


n_features_in: 12
n_estimators: 300
max_depth: 10
min_samples_leaf: 5
classes: [0 1]
n_classes_: 2
feature_importances (len=12):
  feature 2: importance=0.2431
  feature 9: importance=0.1136
  feature 10: importance=0.1099
  feature 3: importance=0.1054
  feature 8: importance=0.1004
  feature 7: importance=0.0960
  feature 11: importance=0.0926
  feature 5: importance=0.0293
  feature 6: importance=0.0285
  feature 4: importance=0.0278


### **pkl** to **onnx** conversion

In [3]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Exemple : si ton modèle prend 12 features
n_features = 12

initial_type = [("input", FloatTensorType([None, n_features]))]
onnx_model = convert_sklearn(rf_model, initial_types=initial_type)

with open("rf_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())


### ONNX checking integrity against pkl model

In [4]:
import joblib
import onnxruntime as ort
import numpy as np
from sklearn.metrics import accuracy_score

# === 1. Charger ton modèle sklearn (.pkl) ===
rf_model = joblib.load(pkl_model_path)

# === 2. Charger le modèle ONNX ===
sess = ort.InferenceSession("rf_model.onnx")
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

# === 3. Créer ou charger un jeu de test ===
# (ici un exemple aléatoire avec 12 features)
X_test = np.random.rand(100, 12).astype(np.float32)

# === 4. Prédiction du modèle original ===
y_pred_sklearn = rf_model.predict(X_test)

# === 5. Prédiction du modèle ONNX ===
y_pred_onnx = sess.run([output_name], {input_name: X_test})[0]

# Certains modèles ONNX renvoient un array de shape (n, 1) → aplatissons-le
y_pred_onnx = np.squeeze(y_pred_onnx)

# === 6. Comparaison ===
equal = np.allclose(y_pred_sklearn, y_pred_onnx, atol=1e-6)
accuracy = accuracy_score(y_pred_sklearn, y_pred_onnx)

print("🔍 Prédictions identiques :", equal)
print("🎯 Accuracy sklearn vs ONNX :", accuracy)
print("Exemples :")
for i in range(5):
    print(f"{i}: sklearn={y_pred_sklearn[i]}, onnx={y_pred_onnx[i]}")


🔍 Prédictions identiques : True
🎯 Accuracy sklearn vs ONNX : 1.0
Exemples :
0: sklearn=0, onnx=0
1: sklearn=0, onnx=0
2: sklearn=0, onnx=0
3: sklearn=1, onnx=1
4: sklearn=0, onnx=0


/home/maxime/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import joblib

model = joblib.load(pkl_model_path)

print("➡️ Nombre de features attendues :", model.n_features_in_)

# Si le modèle a été entraîné sur un DataFrame :
if hasattr(model, "feature_names_in_"):
    print("➡️ Noms des features :", model.feature_names_in_)

➡️ Nombre de features attendues : 12
➡️ Noms des features : ['rsi14' 'macd_hist' 'atr14' 'stoch_k' 'stoch_d' 'time_sin' 'time_cos'
 'ema20_1m' 'ema50_1m' 'ema_slope_1m' 'bb_width_1m' 'bb_percB_1m']
